# Jamii Afya Phase 04 — Falcon-H1-1.5B-Deep-Instruct LoRA PROBE (GPU)
Cheap responsiveness test, NOT the full program. Success bar: medmcqa up
materially (ideally +8..12), arc_easy regression <= 2-3pts, zero critical
safety failures on held-out variants, dispositions reached in budget.
Diagnose misses as data / optimization / LoRA-capacity / model-capacity.
Branch: research/edge35-adaptive-streaming (must be pushed to GitHub first).


In [ ]:
import subprocess, sys
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
OUT = WORK / 'phase04-results'
BRANCH = 'research/edge35-adaptive-streaming'
def run(command, cwd=None, log=None):
    print('+', ' '.join(str(c) for c in command), flush=True)
    r = subprocess.run([str(c) for c in command], cwd=cwd, text=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, check=False)
    print(r.stdout[-3000:], flush=True)
    if log:
        Path(log).parent.mkdir(parents=True, exist_ok=True)
        Path(log).write_text(r.stdout, encoding='utf-8')
    if r.returncode:
        raise RuntimeError(f'exit {r.returncode}: {command}')
    return r
OUT.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH,
         'https://github.com/qeinstein/adtc-llm-limited-hardware.git', str(REPO)])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-dev.txt'],
    cwd=REPO, log=OUT / 'pip_install.log')
run([sys.executable, '-c', 'import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)'])
print('repo sha:', run(['git', 'rev-parse', 'HEAD'], cwd=REPO).stdout.strip())


In [ ]:
# MCQA replay (small: probe must stay cheap) + probe row counts.
run([sys.executable, 'scripts/build_accuracy_sft.py', '--max-per-dataset', '2000'],
    cwd=REPO, log=OUT / 'build_mcqa.log')
run([sys.executable, '-c',
     'import json;'
     'print("probe_sft:", len(json.load(open("data/falcon_probe_sft.json"))));'
     'print("clinical_replay:", len(json.load(open("data/medical_lora_dataset.json"))));'
     'print("heldout:", len(json.load(open("docs/research/falcon_probe_heldout.json"))["prompts"]))'],
    cwd=REPO)


In [ ]:
# Probe: 38 safety/disposition rows + 80 clinical replay + small MCQA replay.
# 3 epochs on ~120 chat rows is minutes on GPU; MCQA replay guards generality.
run([sys.executable, 'scripts/train_lora.py',
     '--base_model', 'tiiuae/Falcon-H1-1.5B-Deep-Instruct',
     '--clinical_file', 'data/falcon_probe_sft.json', 'data/medical_lora_dataset.json',
     '--epochs', '3', '--batch_size', '4', '--grad_accum', '8',
     '--max_len', '512', '--save_steps', '100',
     '--output_dir', str(REPO / 'output' / 'falcon-probe-lora')],
    cwd=REPO, log=OUT / 'train.log')


In [ ]:
import os
# Merge -> GGUF f16 -> Q4_K_M (matches the stock baseline quant; no imatrix for probe).
MERGED = REPO / 'output' / 'falcon-probe-merged'
run([sys.executable, '-c',
     'import sys, torch;'
     'from peft import PeftModel;'
     'from transformers import AutoModelForCausalLM, AutoTokenizer;'
     'base, lora, out = sys.argv[1], sys.argv[2], sys.argv[3];'
     'tok = AutoTokenizer.from_pretrained(base, trust_remote_code=True);'
     'm = AutoModelForCausalLM.from_pretrained(base, torch_dtype=torch.float16, device_map="cpu", trust_remote_code=True);'
     'm = PeftModel.from_pretrained(m, lora).merge_and_unload();'
     'm.save_pretrained(out); tok.save_pretrained(out); print("merged ->", out)',
     'tiiuae/Falcon-H1-1.5B-Deep-Instruct',
     str(REPO / 'output' / 'falcon-probe-lora'), str(MERGED)],
    cwd=REPO, log=OUT / 'merge.log')
LLAMA = WORK / 'llama.cpp'
if not (LLAMA / 'build' / 'bin' / 'llama-quantize').exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA)])
    run(['cmake', '-B', str(LLAMA / 'build'), '-S', str(LLAMA),
         '-DCMAKE_BUILD_TYPE=Release', '-DGGML_NATIVE=ON'])
    run(['cmake', '--build', str(LLAMA / 'build'), '--config', 'Release',
         '-j4', '--target', 'llama-quantize'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=LLAMA)
F16 = OUT / 'falcon-probe-f16.gguf'
QKM = OUT / 'falcon-probe-Q4_K_M.gguf'
run([sys.executable, str(LLAMA / 'convert_hf_to_gguf.py'), str(MERGED),
     '--outfile', str(F16), '--outtype', 'f16'], log=OUT / 'convert.log')
run([str(LLAMA / 'build' / 'bin' / 'llama-quantize'), str(F16), str(QKM), 'Q4_K_M'],
    log=OUT / 'quantize.log')
print('Q4_K_M bytes:', QKM.stat().st_size, flush=True)


In [ ]:
# Eval: MedMCQA-500 + ARC-200 (same harness as baseline) + held-out generations.
run([sys.executable, '-m', 'pip', 'install', '-q', 'llama-cpp-python==0.3.16'],
    log=OUT / 'pip_llama.log')
QKM = OUT / 'falcon-probe-Q4_K_M.gguf'
run([sys.executable, 'scripts/mcq_eval.py', '--model', str(QKM),
     '--task', 'medmcqa', '--limit', '500'], cwd=REPO, log=OUT / 'eval_medmcqa.log')
run([sys.executable, 'scripts/mcq_eval.py', '--model', str(QKM),
     '--task', 'arc_easy', '--limit', '200'], cwd=REPO, log=OUT / 'eval_arc.log')
run([sys.executable, 'scripts/probe_generations.py', '--model', str(QKM),
     '--battery', 'docs/research/falcon_probe_heldout.json',
     '--out-dir', str(OUT / 'gen_heldout')], cwd=REPO, log=OUT / 'gen_heldout.log')
run([sys.executable, 'scripts/probe_generations.py', '--model', str(QKM),
     '--battery', 'docs/research/falcon_baseline_prompts.json',
     '--out-dir', str(OUT / 'gen_battery')], cwd=REPO, log=OUT / 'gen_battery.log')
print('DONE — see phase04-results/')
